<a href="https://colab.research.google.com/github/sidhu2690/MARL/blob/main/Meekle_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import hashlib
import itertools
import random
import time
from collections import defaultdict

NUM_LEAVES = 2048

def h(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()


def entry_hash(entry_id, xy):
    return h(f"{entry_id}:{xy[0]:.6f}:{xy[1]:.6f}".encode())

In [50]:
class MerkleTree:

  def __init__(self, entries: dict):
    self.entries = entries
    self.buckets = defaultdict(list)
    for eid in entries:
      idx = int.from_bytes(h(str(eid).encode()), "big") % NUM_LEAVES
      self.buckets[idx].append(eid)

    leaves = []
    for i in range(NUM_LEAVES):
      ids = sorted(self.buckets.get(i, []))
      if ids:
        blob = b"".join(entry_hash(eid, entries[eid]) for eid in ids)
        leaves.append(h(blob))
      else:
        leaves.append(b"\x00" * 32)

    self.levels = [leaves]
    level = leaves
    while len(level) > 1:
      level = [h(level[i] + level[i+1]) for i in range(0, len(level), 2)]
      self.levels.append(level)
    self.root = self.levels[-1][0]

  def children(self, level, index):
    child_level = self.levels[level - 1]
    return child_level[2 * index], child_level[2*index + 1]

In [51]:
def find_diff_leaves(tA: MerkleTree, tB: MerkleTree):
    if tA.root == tB.root:
        return []
    diffs = []
    stack = [(len(tA.levels) - 1, 0)]
    while stack:
        level, idx = stack.pop()
        if level == 0:
            diffs.append(idx)
            continue
        a0, a1 = tA.children(level, idx)
        b0, b1 = tB.children(level, idx)
        if a0 != b0:
            stack.append((level - 1, 2 * idx))
        if a1 != b1:
            stack.append((level - 1, 2 * idx + 1))
    return diffs

In [52]:
def sync(robot_a, robot_b):
    start = time.perf_counter()

    tree_a = MerkleTree(robot_a.data)
    tree_b = MerkleTree(robot_b.data)
    diff_leaves = find_diff_leaves(tree_a, tree_b)

    exchanged = 0
    for leaf in diff_leaves:
        ids_a = set(tree_a.buckets.get(leaf, []))
        ids_b = set(tree_b.buckets.get(leaf, []))
        for eid in ids_a - ids_b:
            robot_b.data[eid] = robot_a.data[eid]
            exchanged += 1
        for eid in ids_b - ids_a:
            robot_a.data[eid] = robot_b.data[eid]
            exchanged += 1

    elapsed = time.perf_counter() - start
    return elapsed, exchanged

In [53]:
class Robot:
    def __init__(self, robot_id, n_samples=1000, seed=None):
        rng = random.Random(seed)
        self.id = robot_id
        self.data = {
            f"{robot_id}-{i}": (rng.uniform(0, 100), rng.uniform(0, 100))
            for i in range(n_samples)
        }

In [54]:
robots = [Robot(f"R{i}", n_samples=1000, seed=i) for i in range(5)]

print(f"{'Robot':<6}{'Entries before sync':>22}")
for r in robots:
  print(f"{r.id:<6}{len(r.data):>22}")

Robot    Entries before sync
R0                      1000
R1                      1000
R2                      1000
R3                      1000
R4                      1000


In [55]:
total_sync_time = 0.0
total_exchanged = 0

print("\nPairwise sync log:")
for a, b in itertools.combinations(robots, 2):
  elapsed, exchanged = sync(a, b)
  total_sync_time += elapsed
  total_exchanged += exchanged
  print(f"  {a.id} <-> {b.id}: {elapsed*1000:7.3f} ms, {exchanged:4d} entries exchanged")


Pairwise sync log:
  R0 <-> R1:  33.105 ms, 2000 entries exchanged
  R0 <-> R2:  29.060 ms, 3000 entries exchanged
  R0 <-> R3:  33.436 ms, 4000 entries exchanged
  R0 <-> R4:  38.532 ms, 5000 entries exchanged
  R1 <-> R2:  36.072 ms, 1000 entries exchanged
  R1 <-> R3:  56.681 ms, 1000 entries exchanged
  R1 <-> R4:  65.361 ms, 1000 entries exchanged
  R2 <-> R3:  46.391 ms, 1000 entries exchanged
  R2 <-> R4:  56.465 ms, 1000 entries exchanged
  R3 <-> R4:  54.681 ms, 1000 entries exchanged


In [56]:
roots = {r.id: MerkleTree(r.data).root for r in robots}
all_same = len(set(roots.values())) == 1

print(f"\n{'Robot':<6}{'Entries after sync':>22}")
for r in robots:
  print(f"{r.id:<6}{len(r.data):>22}")


Robot     Entries after sync
R0                      5000
R1                      5000
R2                      5000
R3                      5000
R4                      5000


In [57]:
print(f"\nAll robots converged (identical root hash): {all_same}")
print(f"Total sync time (5 robots, {len(list(itertools.combinations(robots,2)))} pairs): "
          f"{total_sync_time*1000:.3f} ms")
print(f"Total entries exchanged: {total_exchanged}")


All robots converged (identical root hash): True
Total sync time (5 robots, 10 pairs): 449.783 ms
Total entries exchanged: 20000
